In [6]:
import sys

print(sys.executable)
print(sys.version)

/opt/anaconda3/bin/python
3.13.9 | packaged by Anaconda, Inc. | (main, Oct 21 2025, 19:11:29) [Clang 20.1.8 ]


In [7]:
%pip install requests pandas python-dotenv matplotlib

Note: you may need to restart the kernel to use updated packages.


In [8]:
%pip install twelvedata

Note: you may need to restart the kernel to use updated packages.


In [9]:
import os
import datetime as dt
import requests
import pandas as pd
from dotenv import load_dotenv
from twelvedata import TDClient

In [10]:
load_dotenv("api_key_file.env")

Alpha_Vantage_API_KEY = os.getenv("ALPHAVANTAGE_API_KEY")
Tiingo_API_TOKEN = os.getenv("TIINGO_API_KEY")
Twelve_Data_API_KEY = os.getenv("TWELVEDATA_API_KEY")
POLYGON_API_KEY = os.getenv("POLYGON_API_KEY")

key_status = {
    "Alpha Vantage": bool(Alpha_Vantage_API_KEY),
    "Tiingo": bool(Tiingo_API_TOKEN),
    "Twelve Data": bool(Twelve_Data_API_KEY),
    "Polygon": bool(POLYGON_API_KEY),
}
print(key_status)

{'Alpha Vantage': True, 'Tiingo': True, 'Twelve Data': True, 'Polygon': True}


In [11]:
Alpha_Vantage_symbol = "AAPL"
Alpha_Vantage_url = "https://www.alphavantage.co/query"

Alpha_Vantage_params = {
    "function": "TIME_SERIES_DAILY",
    "symbol": Alpha_Vantage_symbol,
    "outputsize": "compact",
    "apikey": Alpha_Vantage_API_KEY,
}

Alpha_Vantage_response = requests.get(
    Alpha_Vantage_url,
    params=Alpha_Vantage_params,
    timeout=30,
)

Alpha_Vantage_data = Alpha_Vantage_response.json()

print(Alpha_Vantage_response.status_code)
print(Alpha_Vantage_data.keys())

200
dict_keys(['Meta Data', 'Time Series (Daily)'])


In [12]:
alpha_series = Alpha_Vantage_data.get("Time Series (Daily)")

if not alpha_series:
    raise RuntimeError("No Alpha Vantage daily data returned")

alpha_df = pd.DataFrame.from_dict(
    alpha_series,
    orient="index",
)

alpha_df.index = pd.to_datetime(alpha_df.index)
alpha_df = alpha_df.sort_index()

alpha_df = alpha_df.rename(columns={
    "1. open": "open",
    "2. high": "high",
    "3. low": "low",
    "4. close": "close",
    "5. volume": "volume",
})

In [13]:
STANDARD_COLUMNS = [
    "open",
    "high",
    "low",
    "close",
    "adj_close",
    "volume",
]

alpha_df["adj_close"] = alpha_df["close"]
alpha_df = alpha_df[STANDARD_COLUMNS].copy()
alpha_df = alpha_df.apply(pd.to_numeric, errors="coerce")

display(alpha_df.tail())

,open,high,low,close,adj_close,volume
2026-07-20,333.505,333.7100,323.6800,326.59,326.59,53468008
2026-07-21,323.130,329.6000,322.2204,327.74,327.74,41338917
2026-07-22,327.870,328.9995,323.3400,325.89,325.89,38755929
2026-07-23,321.730,323.3000,319.3500,321.66,321.66,40840778
2026-07-24,321.790,334.3700,321.6200,333.02,333.02,47489415


In [14]:
from twelvedata import TDClient

Twelve_Data_symbol = "AAPL"
Twelve_Data_outputsize = 5000
Twelve_Data_timezone = "America/New_York"

td = TDClient(apikey=Twelve_Data_API_KEY)

Twelve_Data_ts = td.time_series(
    symbol=Twelve_Data_symbol,
    interval="1day",
    outputsize=Twelve_Data_outputsize,
    timezone=Twelve_Data_timezone,
)

twelve_df = Twelve_Data_ts.as_pandas()

print(type(twelve_df))
display(twelve_df.head())

<class 'pandas.core.frame.DataFrame'>


,open,high,low,close,volume
datetime,,,,,
2026-07-24,324.37500,334.37000,321.62000,333.07001,47402209
2026-07-23,321.73001,323.29999,319.35001,321.66000,40840800
2026-07-22,327.87000,329.00000,323.34000,325.89001,38755900
2026-07-21,323.13000,329.60001,322.22000,327.73999,41338900
2026-07-20,333.51001,333.70999,323.67999,326.59000,53468000


In [15]:
ticker = "AAPL"

start_date = "2024-01-01"
end_date = "2024-07-25"

url = f"https://api.polygon.io/v2/aggs/ticker/{ticker}/range/1/day/{start_date}/{end_date}"


params = {
    "adjusted": "true",
    "sort": "asc",
    "limit": 100,
    "apiKey": POLYGON_API_KEY
}

In [16]:
response = requests.get(
    url,
    params=params,
    timeout=30
)

response.raise_for_status()

POLYGON_data = response.json()

In [17]:
print(POLYGON_data.keys())

dict_keys(['ticker', 'queryCount', 'resultsCount', 'adjusted', 'results', 'status', 'request_id', 'count'])


In [18]:
results = POLYGON_data.get("results", [])

if not results:
    raise RuntimeError(
        f"Polygon: no results returned: {POLYGON_data}"
    )


polygon_raw_df = pd.DataFrame(results)


polygon_raw_df["date"] = pd.to_datetime(
    polygon_raw_df["t"],
    unit="ms"
)


polygon_raw_df = (
    polygon_raw_df
    .set_index("date")
    .sort_index()
)


polygon_df = polygon_raw_df.rename(
    columns={
        "o": "open",
        "h": "high",
        "l": "low",
        "c": "close",
        "v": "volume"
    }
)

In [20]:
polygon_df["adj_close"] = polygon_df["close"]
polygon_df = polygon_df[STANDARD_COLUMNS].copy()

In [21]:
polygon_df = polygon_df.apply(
    pd.to_numeric,
    errors="coerce"
)

In [22]:
display(polygon_df.head())

,open,high,low,close,adj_close,volume
date,,,,,,
2024-07-25 04:00:00,218.93,220.85,214.62,217.49,217.49,51391199.0
